# Spectral clustering of a plant point cloud — full-matrix prototype

*Converted from the exploratory script `Main.py` (kept faithful to the original approach).*

*Archived source: `Main.py` @ commit `fe1a7fa` — restore with `git show fe1a7fa:Main.py`.*


## Goal

Given a 3D scan of a plant, **automatically group the points into meaningful
parts (stems, leaves) from geometry alone**, with no manual labelling.

- **For biologists:** the plant's organs are not labelled in a raw 3D scan; we
  want the computer to discover them as clusters of points that belong
  together, purely from their spatial arrangement.
- **For computer scientists:** this is an *unsupervised* clustering problem.
  The very first prototype used a **full pairwise matrix**: every point is
  compared to every other point, which is why it only scales to small clouds.

## Implementation overview

1. Load a 3D point cloud.
2. Build a **similarity matrix**: how "close" every pair of points is
   (reciprocal of the Euclidean distance), then keep only the strongest
   connections by thresholding.
3. Derive the **graph Laplacian** \(L = D - W\) (degree matrix minus the
   similarity / adjacency matrix).
4. Compute its **eigenvectors** with `np.linalg.eigh`.
5. Run **k-means** in the space of the \(k\) smallest eigenvectors — this is the
   essence of *spectral clustering*: clustering in a low-dimensional
   eigen-space often separates the plant's natural parts better than
   clustering the raw 3D coordinates.

> The original script loaded a now-missing `arabi_ascii_segm_double.ply` cloud
> and used the obsolete `open3d.read_point_cloud` and `vplants.cellcomplex`
> APIs. Here we load a shipped cloud with the current `open3d` API. Because the
> full-matrix method is \(O(n^2)\) in memory and time, we subsample to a few
> thousand points so the demo runs quickly (see the motivation for the
> graph-based method in the next notebook).


In [ ]:
import numpy as np
import open3d as o3d
import scipy.spatial as sps
import scipy.cluster as spc
import matplotlib.pyplot as plt

### 1. Load the point cloud

We read a *Chenopodium album* (goosefoot / lamb's quarters) scan. The raw
cloud has tens of thousands of points; we keep a random subset so the
full-matrix computation stays tractable in a notebook.

In [ ]:
pcd = o3d.io.read_point_cloud("../data/chenos/cheno_3.ply")
print(pcd)

In [ ]:
points = np.asarray(pcd.points)
print("full cloud size:", points.shape)

In [ ]:
# Subsample for the O(n^2) full-matrix prototype.
if points.shape[0] > 3000:
    rng = np.random.default_rng(0)
    idx = rng.choice(points.shape[0], 3000, replace=False)
    points = points[idx]
print("subsampled size:", points.shape)

### 2. Similarity matrix

`scipy.spatial.distance_matrix` computes the Euclidean distance between
*every pair* of points (the expensive part). The **similarity** between two
points is the reciprocal of their distance: closer points are more similar.
Distances of zero (a point with itself) are handled so the diagonal becomes 0,
and a threshold keeps only the strongest (closest) connections, sparsifying
the matrix — the same logic as the original `Main.py`.

In [ ]:
# Pairwise Euclidean distance matrix.
d = sps.distance_matrix(points, points, p=2)

In [ ]:
# Similarity = reciprocal of distance (closer = more similar).
d[d == 0] = -1              # avoid division by zero on the diagonal
sim = 1.0 / d
sim[sim < 0] = 0            # restore the diagonal to 0

In [ ]:
# Sparsify: keep only connections between very close points.
# (The original script thresholded at 700 for a differently-scaled cloud;
#  here we use 2.0, the scale of the shipped Chenopodium cloud.)
threshold_dist = 2.0        # keep points closer than 2.0 (cloud units)
sim[sim < 1.0 / threshold_dist] = 0

In [ ]:
sparsity = 1.0 - (np.count_nonzero(sim) / float(sim.size))
print(f"matrix density: {1 - sparsity:.4f}  (1.0 = fully dense)")

### 3. Visualise the similarity matrix

Each pixel is the similarity between two points. Because the plant's points
are ordered arbitrarily, the matrix looks mostly dark; the bright stripes
reveal groups of mutually-close points — the first hint of the organs.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(sim[:200, :200], cmap="viridis")
ax.set_title("Similarity matrix (first 200 points)")
fig.colorbar(im, ax=ax)
plt.show()

### 4. Graph Laplacian \(L = D - W\)

The **degree matrix** \(D\) holds on its diagonal the sum of each row of the
similarity matrix (a measure of how connected each point is). The **graph
Laplacian** is

\[ L = D - W \]

where \(W\) is the similarity (weight) matrix. The Laplacian encodes the
graph structure and is the object whose eigenvectors we will use to find
natural clusters.

In [ ]:
size = sim.shape[0]

# Degree matrix D: sum of each row placed on the diagonal.
degree = np.sum(sim, axis=1, dtype=float).reshape(1, size)
degmat = np.identity(size)
di = np.diag_indices(size)
degmat[di] = degree

In [ ]:
# Graph Laplacian L = D - W.
L = degmat - sim

### 5. Eigenvectors + k-means (spectral clustering)

`np.linalg.eigh` diagonalises the real symmetric matrix \(L\), giving its
eigenvalues (sorted ascending) and eigenvectors. We keep the \(k\) **smallest**
eigenvectors — these capture the large-scale structure of the graph. Running
k-means in this reduced eigen-space assigns each point to a cluster.

In [ ]:
k = 2
eigenval, eigenvec = np.linalg.eigh(L)
keigenvec = eigenvec[:, :k]

In [ ]:
centroids, label = spc.vq.kmeans2(keigenvec, k, minit="points", missing="warn")
labels = label.reshape(-1, 1)
print("cluster sizes:", np.bincount(labels[:, 0].astype(int)))

### 6. Visualise the result

Each point is coloured by its cluster. Even from this naive full-matrix
prototype, the two clusters start to separate the plant's structure.

In [ ]:
fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, projection="3d")
sc = ax.scatter(points[:, 0], points[:, 1], points[:, 2], c=labels[:, 0], cmap="tab10", s=3)
fig.colorbar(sc, ax=ax, label="cluster")
ax.set_title("Full-matrix spectral clustering (k=2)")
plt.show()

## Takeaway

This prototype proves the *spectral* idea works, but it has two serious
limits:

1. **Memory/time:** the full \(O(n^2)\) matrix does not scale to real plants.
2. **Geometry model:** comparing every point to every other ignores the
   plant's chain-like topology (a stem connecting to leaves).

The next notebooks fix both by building a **sparse nearest-neighbour
(Riemannian) graph** and using **graph eigenvectors** instead of a dense
matrix — and finally by reading the *gradient of the Fiedler vector* to find
the branch points.